# 1. Install libraries

In [ ]:
# Fix environment for Colab
!pip uninstall -y numpy scipy scikit-learn
!pip install -q numpy==2.0.2 scipy==1.14.1 scikit-learn==1.6.1

import numpy as np
import scipy
import sklearn

print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("sklearn:", sklearn.__version__)

# 2. Clone Github repository

In [ ]:
!rm -rf BTL_ML
!git clone https://github.com/Hoang-Viet-Tran/BTL_ML.git
%cd BTL_ML

# 3. Add folder "modules" into Python path

In [ ]:
import sys
sys.path.insert(0, "/content/BTL_ML/modules")

# 4. Download data

In [ ]:
import os

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

BASE_URL = "https://datasets.imdbws.com/"

files = [
    "title.basics.tsv.gz",
    "title.ratings.tsv.gz"
]

for f in files:
    url = BASE_URL + f
    print("Downloading", f)
    !wget -q -O {DATA_DIR}/{f} {url}

# 5. Import modules

In [ ]:
from data_utils import load_raw_data
from preprocess import preprocess_df
from features import extract_features, save_features

# 6. Load and preprocess data

In [ ]:
df = load_raw_data("/content/BTL_ML/data")
print("Dataset shape:", df.shape)

df = preprocess_df(df)
print("After preprocessing:", df.shape)

df.head()

# 7. Merge datasets to create modeling dataframe

In [ ]:
# Combine IMDb tables into single dataframe for modeling
basics = dfs[0]
ratings = dfs[1]

df = basics.merge(ratings, on="tconst", how="inner")

print("Final merged df shape:", df.shape)

# 8. Feature extraction

In [ ]:
df = preprocess_df(df)

LABEL_COLUMN = "is_success"

X, y = extract_features(df, label_column=LABEL_COLUMN)

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# 9. Save features

In [ ]:
save_features(X, y, out_dir="/content/BTL_ML/features")

print("✅ Features saved into /features/")
!ls features

# 10. Train model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_val)[:,1]
auc = roc_auc_score(y_val, y_prob)

print("Validation AUC:", auc)

# 11. Save model

In [ ]:
import joblib

joblib.dump(model, "best_model.joblib")
print("✅ Model saved")

# 12. Demo

In [ ]:
print("=== Demo decision system ===")
samples = [
    [2023, 120, 7.3, 15000],   # borderline
    [2018, 150, 8.9, 300000],  # rất tốt
    [2005, 90, 5.4, 4000],     # rất tệ
    [2022, 110, 7.8, 80000]    # khá ổn
]

for s in samples:
    prob = model.predict_proba([s])[0][1]
    print(f"Input: {s} → Success probability: {prob:.3f}")
    if prob > 0.5:
      print("✅ Should buy this film")
    else:
      print("❌ Should not buy this film")